# Angat Dam: water level and storage volume from SWOT + Sentinel-1
SWOT `SWOT_L2_HR_LakeSP` gives the **water level** (`wse`). The Sentinel-1 surface-water maps (`areas.csv` from `scripts/map_water.py`) give the **area**. Pairing them gives the
area-elevation curve A(h), and the integral of A over level gives the **storage volume**. Needs a free NASA Earthdata login.

In [ ]:
# 0. Setup
import os, sys, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "swot_volume").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if "google.colab" in sys.modules or os.path.exists("/content"):
    get_ipython().run_line_magic("pip", "install -q earthaccess geopandas pandas numpy matplotlib")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from swot_volume import swot, integrate, hypsometry

In [ ]:
# 1. Configuration
BBOX = (121.14302981543655, 14.872088697247067, 121.26233447214564, 15.053187085370627)   # Angat Dam: west, south, east, north
SITE = "angat_dam"
AREAS_CSV = ROOT / "outputs/maps/areas.csv"      # Sentinel-1 water areas from map_water.py
START, END = "2023-08-01", "2025-06-30"
SWOT_DIR = ROOT / "cache/swot"                   # downloaded shapefiles are reused from here
MAX_QUALITY = 1                                  # SWOT quality_f <= this (0 good, 1 suspect)
ALLOW_PARTIAL = False                            # lakes only partly inside the swath
MAX_LAG_DAYS = 6                                 # SWOT <-> Sentinel-1 pairing window
METHOD = "isotonic"                              # or "power"
H_REF, V_REF = None, None                        # level (m) and storage (million m3) at that level -> absolute volume; None = relative

In [ ]:
# 2. SWOT lake observations over the reservoir
paths = sorted(glob.glob(str(SWOT_DIR / "**/*_Obs_*.shp"), recursive=True))
if not paths:
    paths = swot.download(swot.search(BBOX, START, END), str(SWOT_DIR), kind="Obs")
g = swot.read_shapefiles(paths, bbox=BBOX)
obs = swot.observations_from_frames(g, max_quality=MAX_QUALITY, allow_partial=ALLOW_PARTIAL)
obs = obs[(obs["date"] >= START) & (obs["date"] <= END)]
print(len(obs), "usable SWOT passes")
obs.head()

In [ ]:
# 3. Sentinel-1 area, pairing, area-elevation curve and volume
s1 = integrate.s1_daily_area(str(AREAS_CSV), site=SITE)
table, curve = integrate.compute(obs, s1, MAX_LAG_DAYS, METHOD, H_REF, V_REF)
print(f"{table['s1_area_km2'].notna().sum()} of {len(table)} SWOT passes paired; area-curve RMSE {integrate.rmse_of_curve(table):.2f} km2")
table

In [ ]:
# 4. Plots
t = table.assign(date=pd.to_datetime(table["date"]))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].errorbar(t["date"], t["wse_m"], yerr=t["wse_u_m"], fmt="o-"); ax[0].set(title="SWOT water level", ylabel="m (EGM2008)")
p = t.dropna(subset=["s1_area_km2"]); hs = np.linspace(t["wse_m"].min(), t["wse_m"].max(), 100)
ax[1].scatter(p["wse_m"], p["s1_area_km2"], label="SWOT level / S1 area"); ax[1].plot(hs, curve.area(hs), "r", label="fitted A(h)")
ax[1].set(title="Area-elevation curve", xlabel="level (m)", ylabel="area (km2)"); ax[1].legend()
ax[2].plot(t["date"], t["volume_mcm"], "o-"); ax[2].set(title=f"Storage ({t['volume_basis'].iloc[0]})", ylabel="million m3")
for a in ax: a.grid(alpha=.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
# 5. Save
OUT = ROOT / "outputs/swot_volume"; OUT.mkdir(parents=True, exist_ok=True)
table.to_csv(OUT / "level_area_volume.csv", index=False)
print("wrote", OUT / "level_area_volume.csv")